# Toy GraphCast-Style Skill Demo

This notebook creates a **small synthetic dataset** to demonstrate the evaluation workflow used in weather-forecast model comparisons (skill vs lead time), without bundling large reanalysis datasets in the repo.

**You will:**
- Generate synthetic absolute errors for a baseline model and a “GraphCast-like” model.
- Compute MAE by lead time.
- Run a paired t-test (baseline error > GraphCast-like error).
- Plot a skill curve.

> Educational scaffold: replace the synthetic generator with ERA5 / real forecast outputs when ready.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

SEED = 7
N_SAMPLES = 200
LEAD_HOURS = [6, 12, 24, 48, 72, 96, 120, 144, 168, 192, 216, 240]


In [ ]:
rng = np.random.default_rng(SEED)

def make_synthetic_errors(n: int, lead_h: int, model: str) -> np.ndarray:
    base = 1.0 + 0.010 * lead_h
    noise = rng.lognormal(mean=0.0, sigma=0.35, size=n)
    if model == "baseline":
        scale = 1.15
    elif model == "graphcast_like":
        scale = 0.95
    else:
        raise ValueError(model)
    return scale * base * noise

rows = []
for lead_h in LEAD_HOURS:
    e_base = make_synthetic_errors(N_SAMPLES, lead_h, "baseline")
    e_gc = make_synthetic_errors(N_SAMPLES, lead_h, "graphcast_like")
    for i in range(N_SAMPLES):
        rows.append({
            "sample_id": i,
            "lead_hours": lead_h,
            "baseline_abs_error": float(e_base[i]),
            "graphcast_like_abs_error": float(e_gc[i]),
        })

df = pd.DataFrame(rows)
df["error_diff"] = df["baseline_abs_error"] - df["graphcast_like_abs_error"]
df.head()


In [ ]:
summary = (
    df.groupby("lead_hours", as_index=False)
      .agg(
          baseline_mae=("baseline_abs_error", "mean"),
          graphcast_like_mae=("graphcast_like_abs_error", "mean"),
          diff_mean=("error_diff", "mean"),
          diff_std=("error_diff", "std"),
      )
      .sort_values("lead_hours")
)
summary


In [ ]:
ttests = []
for lead_h, sub in df.groupby("lead_hours"):
    t, p = stats.ttest_rel(
        sub["baseline_abs_error"].to_numpy(),
        sub["graphcast_like_abs_error"].to_numpy(),
        alternative="greater",
    )
    ttests.append({"lead_hours": int(lead_h), "t_stat": float(t), "p_value": float(p)})

ttests = pd.DataFrame(ttests).sort_values("lead_hours")
ttests


In [ ]:
x = summary["lead_hours"].to_numpy()
plt.figure(figsize=(10, 5))
plt.plot(x, summary["baseline_mae"], marker="o", label="Baseline (MAE)")
plt.plot(x, summary["graphcast_like_mae"], marker="o", label="GraphCast-like (MAE)")
plt.title("Toy skill curve (lower MAE is better)")
plt.xlabel("Lead time (hours)")
plt.ylabel("Mean absolute error (a.u.)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()
